In [44]:
from sentropy import sentropy

## Row-by-row streaming
Here we demonstrate that streaming the matrix row-by-row on the fly is more efficient than calculating an entire similarity matrix separately.  
The example used here is the BloodMNIST dataset (@medmnistv2), which should take ~30s to download.  
Device used: Mac Mini

In [45]:
import numpy as np
from medmnist import BloodMNIST
from joblib import Parallel, delayed
from sentropy import sentropy
MEASURES =["alpha", "rho", "beta", "gamma", "normalized_alpha",\
"normalized_rho", "normalized_beta", "rho_hat", "beta_hat"]

In [46]:
def rmse_sim(vec1, vec2):
    '''
    Similarity function used by sentropy (exp(-RMSE)).
    '''
    rmse = np.sqrt(np.mean((vec1 - vec2) ** 2))
    return np.exp(-rmse)

def compute_similarity(i, j, X):
    '''
    Parallelization function
    '''
    return i, j, rmse_sim(X[i], X[j])


def get_similarity_matrix(X, n_jobs=-1):
    '''
    Parallelized calculation of similarity matrix
    '''
    n = len(X)
    similarity = np.eye(n)

    # Generate upper-triangle index pairs
    pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]

    results = Parallel(n_jobs=n_jobs)(
        delayed(compute_similarity)(i, j, X)
        for i, j in pairs
    )

    for i, j, sim in results:
        similarity[i, j] = sim
        similarity[j, i] = sim

    return similarity


def block_based(X):
    '''
    calculating similarity matrix separately
    '''
    similarity = get_similarity_matrix(X, n_jobs=-1)
    sentropy(label_ind, similarity=similarity, q=[0, 1, 1.5, np.inf], measure=MEASURES)


def row_based(X):
    '''
    row-by-row calculation
    '''
    sentropy(label_ind, similarity=rmse_sim, sfargs=X, q=[0, 1, 1.5, np.inf], measure=MEASURES)

In [47]:
# loading dataset
test_dataset = BloodMNIST(split="test", download=True)

# Extract images
X = np.array([np.array(img) for img, label in test_dataset])

# Flatten images: (N, H, W, C) -> (N, H*W*C)
X = X.reshape(len(X), -1).astype(np.float32)

# creating class (label) dictionary
labels = np.array([x[0] for _, x in test_dataset])
label_ind = {}
for label in set(labels):
    label_ind[label] = labels == label

In [48]:
%%timeit
block_based(X)

9.18 s ± 148 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [49]:
%%timeit
row_based(X)

5.41 s ± 191 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Performance benchmark
Here we benchmark the speed for running sentropy with different settings

In [ ]:
import numpy as np
MEASURES =["alpha", "rho", "beta", "gamma", "normalized_alpha",\
"normalized_rho", "normalized_beta", "rho_hat", "beta_hat"]

big_counts = np.random.randint(101, size=(10000,100))

n = 10000
indices = np.triu_indices(n, k=1)
values = np.random.rand(len(indices[0]))
big_sim_matrix = np.zeros((n, n))
big_sim_matrix[indices] = values
big_sim_matrix = big_sim_matrix + big_sim_matrix.T
np.fill_diagonal(big_sim_matrix, 1.0)

### Device 1: Mac Mini (Apple M4 Pro chip, 64 GB memory)

In [6]:
%%timeit
sentropy(big_counts, similarity=big_sim_matrix, q=[0,1, 1.5, np.inf], measure=MEASURES)

941 ms ± 59.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [4]:
%%timeit
sentropy(big_counts, similarity=big_sim_matrix, q=[0,1, 1.5, np.inf], measure=MEASURES, backend='torch')

233 ms ± 3.57 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [3]:
%%timeit
sentropy(big_counts, similarity=big_sim_matrix, q=[0,1, 1.5, np.inf], measure=MEASURES, backend='torch', device='mps')

145 ms ± 29.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


### Device 2: Ubuntu (AMD Ryzen Threadripper 7960X 24-Cores with 2 NVIDIA RTX 6000 Ada Generation GPUs)

In [9]:
%%timeit
sentropy(big_counts, similarity=big_sim_matrix, q=[0,1, 1.5, np.inf], measure=MEASURES)

336 ms ± 26.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
%%timeit
sentropy(big_counts, similarity=big_sim_matrix, q=[0,1, 1.5, np.inf], measure=MEASURES, backend='torch')

237 ms ± 21.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [8]:
%%timeit
sentropy(big_counts, similarity=big_sim_matrix, q=[0,1, 1.5, np.inf], measure=MEASURES, backend='torch',  device='cuda')

91.6 ms ± 305 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Reference
@article{medmnistv2,
    title={MedMNIST v2-A large-scale lightweight benchmark for 2D and 3D biomedical image classification},
    author={Yang, Jiancheng and Shi, Rui and Wei, Donglai and Liu, Zequan and Zhao, Lin and Ke, Bilian and Pfister, Hanspeter and Ni, Bingbing},
    journal={Scientific Data},
    volume={10},
    number={1},
    pages={41},
    year={2023},
    publisher={Nature Publishing Group UK London}
}